# 第三题：实现带有拉普拉斯修正的朴素贝叶斯

实验内容：
1. 叙述拉普拉斯修正的作用
2. 给出使用的数据集
3. 给出实现的代码，要有详细的注释
4. 给出模型评价指标的结果

In [1]:
import numpy as np
from pathlib import Path

# 兼容从项目根目录或 solution 目录启动内核。
# 数据只从 data 目录读取，训练过程中不修改原始文件。
data_path = next(
    (base / "data" / "spambase" / "spambase.data"
     for base in (Path.cwd(), Path.cwd().parent)
     if (base / "data" / "spambase" / "spambase.data").is_file()),
    None,
)
if data_path is None:
    raise FileNotFoundError("请从项目根目录或 solution 目录运行本 Notebook。")
spambase = np.loadtxt(data_path, delimiter=",")
spamx = spambase[:, :57]
spamy = spambase[:, 57].astype(int)
assert spamx.shape == (4601, 57)
assert set(np.unique(spamy)) == {0, 1}
assert np.isfinite(spamx).all() and (spamx >= 0).all()
print(f"Spambase: {spamx.shape[0]} 个样本，{spamx.shape[1]} 个特征")
print(f"正常邮件: {(spamy == 0).sum()}；垃圾邮件: {(spamy == 1).sum()}")


Spambase: 4601 个样本，57 个特征
正常邮件: 2788；垃圾邮件: 1813


## 拉普拉斯修正的作用与数据表示

使用 Spambase 垃圾邮件数据集，最后一列中 `1` 表示垃圾邮件，`0` 表示正常邮件。
本题手写**伯努利朴素贝叶斯**：将每个非负连续特征按 `x_j > 0` 转换成二值特征。
对词频和字符频率，这表示对应词或字符是否出现；对于大写字母统计，表示是否大于零。
二值化规则预先固定，不从测试集估计参数。

若某类中某个事件未出现，极大似然条件概率为零，连乘时会把该类整个得分变成零。
对每种可能事件增加伪计数 $\alpha$ 后，零频事件仍有非零概率；$\alpha=1$ 即拉普拉斯修正。
更大的 $\alpha$ 会使条件概率更接近 $1/2$、类别先验更接近均匀分布。

设类别数为 $K$，该类样本数为 $N_c$，其中第 $j$ 个二值特征为 1 的样本数为 $N_{c,j}$：

$$P(c)=\frac{N_c+\alpha}{N+K\alpha},\qquad
\theta_{c,j}=P(z_j=1\mid c)=\frac{N_{c,j}+\alpha}{N_c+2\alpha}.$$

$$\log P(c,z)=\log P(c)+\sum_j\left[
z_j\log\theta_{c,j}+(1-z_j)\log(1-\theta_{c,j})\right].$$

分母中的 `2α` 来自二值特征的两种取值。预测同时考虑出现和未出现的证据。
以下类的训练和预测均手写完成，sklearn 仅用于划分数据、评价和独立核对。


In [2]:
from sklearn.base import BaseEstimator, ClassifierMixin
from sklearn.utils.validation import check_X_y, check_array, check_is_fitted

class LaplaceBernoulliNB(ClassifierMixin, BaseEstimator):
    """对类别先验和二值条件概率都实施加 alpha 平滑的朴素贝叶斯。"""

    def __init__(self, alpha=1.0, threshold=0.0):
        self.alpha = alpha
        self.threshold = threshold

    def fit(self, X, y):
        # sklearn 校验只处理数组形状和有限值，不参与模型参数估计。
        X, y = check_X_y(X, y, dtype=float)
        if not np.isfinite(self.alpha) or self.alpha <= 0:
            raise ValueError("alpha 必须为正数；alpha=1 是拉普拉斯修正。")
        if not np.isfinite(self.threshold):
            raise ValueError("二值化阈值必须为有限数。")
        self.n_features_in_ = X.shape[1]
        self.classes_, class_index = np.unique(y, return_inverse=True)
        self.class_count_ = np.bincount(class_index).astype(float)
        binary = (X > self.threshold).astype(float)

        # 各行对应类别，各列对应特征；统计每类中该特征出现的次数。
        self.feature_count_ = np.vstack([
            binary[class_index == k].sum(axis=0)
            for k in range(len(self.classes_))
        ])
        self.class_prior_ = (
            (self.class_count_ + self.alpha)
            / (len(y) + self.alpha * len(self.classes_))
        )
        self.feature_prob_ = (
            (self.feature_count_ + self.alpha)
            / (self.class_count_[:, None] + 2 * self.alpha)
        )
        self.class_log_prior_ = np.log(self.class_prior_)
        self.feature_log_prob_ = np.log(self.feature_prob_)
        # log1p(-p) 在 p 较小时比直接计算 log(1-p) 更稳定。
        self.feature_log_absent_ = np.log1p(-self.feature_prob_)
        return self

    def predict_log_proba(self, X):
        from scipy.special import logsumexp

        check_is_fitted(self, "feature_prob_")
        X = check_array(X, dtype=float)
        if X.shape[1] != self.n_features_in_:
            raise ValueError("训练和测试的特征数不一致。")
        binary = (X > self.threshold).astype(float)
        # 一次矩阵乘法计算全部样本在各个类别下的对数联合得分。
        joint = (
            binary @ self.feature_log_prob_.T
            + (1 - binary) @ self.feature_log_absent_.T
            + self.class_log_prior_
        )
        # 减去 logsumexp 归一化，避免先 exp 再求和造成数值下溢。
        return joint - logsumexp(joint, axis=1, keepdims=True)

    def predict_proba(self, X):
        return np.exp(self.predict_log_proba(X))

    def predict(self, X):
        return self.classes_[np.argmax(self.predict_log_proba(X), axis=1)]


## 固定测试集评价

与第二题共用 60%/40% 的划分及随机种子 32。
主实验预先选定 `alpha=1`，查准率、查全率、F1 均以垃圾邮件 `1` 为正类。


In [3]:
import pandas as pd
from IPython.display import display
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score, confusion_matrix,
)
from sklearn.naive_bayes import BernoulliNB

trainX, testX, trainY, testY = train_test_split(
    spamx, spamy, test_size=0.4, random_state=32
)
model = LaplaceBernoulliNB(alpha=1.0).fit(trainX, trainY)
prediction = model.predict(testX)
test_metrics = {
    "accuracy": accuracy_score(testY, prediction),
    "precision": precision_score(testY, prediction, zero_division=0),
    "recall": recall_score(testY, prediction, zero_division=0),
    "f1": f1_score(testY, prediction, zero_division=0),
}
display(pd.DataFrame([test_metrics], index=["LaplaceBernoulliNB"]).round(6))
display(pd.DataFrame(
    confusion_matrix(testY, prediction, labels=[0, 1]),
    index=["真实正常邮件", "真实垃圾邮件"],
    columns=["预测正常邮件", "预测垃圾邮件"],
))

# 默认 sklearn 不平滑类别先验，此处传入相同先验后再核对两种实现。
reference = BernoulliNB(
    alpha=1.0, binarize=0.0, class_prior=model.class_prior_
).fit(trainX, trainY)
np.testing.assert_allclose(
    model.feature_log_prob_, reference.feature_log_prob_, rtol=1e-12, atol=1e-12
)
np.testing.assert_allclose(
    model.predict_log_proba(testX), reference.predict_log_proba(testX), atol=1e-10
)
np.testing.assert_array_equal(prediction, reference.predict(testX))
np.testing.assert_allclose(model.predict_proba(testX).sum(axis=1), 1.0)
assert np.all((model.feature_prob_ > 0) & (model.feature_prob_ < 1))
print("验证通过：条件概率、后验对数概率与相同先验的 sklearn BernoulliNB 一致。")


,accuracy,precision,recall,f1
LaplaceBernoulliNB,0.887018,0.884146,0.814607,0.847953


,预测正常邮件,预测垃圾邮件
真实正常邮件,1053,76
真实垃圾邮件,132,580


验证通过：条件概率、后验对数概率与相同先验的 sklearn BernoulliNB 一致。


In [4]:
# 直接观察平滑如何消除训练中的零频事件。
unsmoothed_prob = model.feature_count_ / model.class_count_[:, None]
unsmoothed_zero_events = int(np.sum(unsmoothed_prob == 0) + np.sum(unsmoothed_prob == 1))
smoothed_zero_events = int(np.sum(model.feature_prob_ == 0) + np.sum(model.feature_prob_ == 1))
print(f"未平滑时零概率的（类别、特征、取值）事件数：{unsmoothed_zero_events}")
print(f"拉普拉斯修正后相同计数：{smoothed_zero_events}")
print(f"平滑后的条件概率范围：[{model.feature_prob_.min():.8f}, {model.feature_prob_.max():.8f}]")
assert smoothed_zero_events == 0


未平滑时零概率的（类别、特征、取值）事件数：6
拉普拉斯修正后相同计数：0
平滑后的条件概率范围：[0.00180614, 0.99939795]


## 平滑强度的补充比较

只在训练集内做分层十折交叉验证，观察不同正数 `alpha` 的表现。
固定测试集不参与本比较，也不根据测试指标选择参数；主实验仍使用题目要求的 `alpha=1`。


In [5]:
from sklearn.model_selection import StratifiedKFold, cross_validate

scoring = {"accuracy": "accuracy", "precision": "precision", "recall": "recall", "f1": "f1"}
alpha_records = []
for alpha in [0.01, 0.1, 1.0, 10.0]:
    scores = cross_validate(
        LaplaceBernoulliNB(alpha=alpha), trainX, trainY,
        cv=StratifiedKFold(n_splits=10, shuffle=True, random_state=42),
        scoring=scoring, error_score="raise",
    )
    alpha_records.append({
        "alpha": alpha,
        **{name: scores[f"test_{name}"].mean() for name in scoring},
    })
alpha_results = pd.DataFrame(alpha_records).set_index("alpha")
display(alpha_results.round(6))


,accuracy,precision,recall,f1
alpha,,,,
0.01,0.885145,0.889333,0.813792,0.849590
0.10,0.884783,0.889234,0.812883,0.849048
1.00,0.885145,0.889303,0.813792,0.849576
10.00,0.881522,0.885491,0.808346,0.844783


## 结果与分析

| 算法 | 准确率 accuracy | 查准率 precision | 查全率 recall | F1 |
| --- | --- | --- | --- | --- |
| LaplaceBernoulliNB（alpha=1） | 0.887018 | 0.884146 | 0.814607 | 0.847953 |

alpha=1 时，固定测试集准确率为 0.887018，查准率为 0.884146，查全率为 0.814607，F1 为 0.847953。训练集原始计数中共有 6 个（类别、特征、取值）零频事件；加一后所有二值条件概率均严格位于 (0, 1)。手写条件概率、归一化对数概率和预测结果已与使用相同类别先验的 sklearn BernoulliNB 核对一致。上方平滑强度比较仅在训练集内部交叉验证，未使用固定测试集选择 alpha。

加一修正避免未见事件把类别得分直接归零，并减小小样本概率估计的极端程度。
二值化舍弃了频率大小，因此本模型的结果同时受到表示方式和分布假设的影响；
不能将它与第二题的指标差异全部归因于平滑。拉普拉斯修正也不保证任何数据集上的指标必然提高。
